# Model

- Split to quadrants.

- Split each quadrant by office.

- Detect Positions location.

---------NEW--------
- Find top row for each column.

    Define as the same height of position/ Month character (月,年, etc).
    
- Crawl down top row of each column.

- Split to wages and names by length of character sequence.

In [1]:
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'

import sys
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')

In [2]:
#Set up Environment
import os,cv2
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')
from Split import HoriPart, VertPart
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from OCR import Clova
api_url='https://1srlcnzf08.apigw.ntruss.com/custom/v1/2412/9a859f9b3a7d952aad17f388d1a445041f8aef0f1eccc6fcce8d3a856272fcbd/general'
secret_key='eEhyV0NGRlRLVXpZVWZnWFNDamRVaWFYZ1NSQ1NKSFI='


os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [3]:
import json, os, cv2
import pandas as pd

Year,Showa='1942','17'
Quality='HQ'

df = pd.read_csv(origin+'/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

# Step A. Set up Page

In [4]:
Page=20
file_path2='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
img=cv2.imread(os.path.join(origin,file_path2))
try:
    if img==None:
        file_path2='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.png'
        img=cv2.imread(os.path.join(origin,file_path2))    
except:
    img=cv2.imread(os.path.join(origin,file_path2))    

cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_Window", 1280, 1440)
cv2.imshow("Resized_Window",img)
cv2.waitKey(0)
cv2.destroyAllWindows()


#Convert to book page
BookPage=2*Page-14
Rows=df[(df['Page']==BookPage)]
NxRows=df[(df['Page']==BookPage+1)]
if len(Rows)==0:
    print('No Office at Right Side. Page'+str(BookPage))
if len(NxRows)==0:
    print('No Office at Left Side. Page'+str(BookPage+1))
        
texts=Vision(img,'zh',client)
textsCLOVA=Clova(origin,Page,api_url,secret_key,Year,Quality)

# Step B. Get Office Info

In [5]:
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')
from Split import VertPart

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from Organize import Filter, ConvertDict
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart

file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
path=os.path.join(origin,file_path)

HoriPoint=HoriPart(img,Page,texts)[0]

HH,WW=img.shape[:2]
VertPoint=1*HH//3

##Right Page
BoxR=Filter(BookPage,texts,HoriPoint)
BoxR=ConvertDict(BoxR)

##Left Page
BoxL=Filter(BookPage+1,texts,HoriPoint)
BoxL=ConvertDict(BoxL)

In [6]:
Dict={}
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from Organize import FilterBox
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from DetectOffice import DetectOffice
LocList=[]

#RightPage
OfficeList=Rows['Office'].tolist()
for Office in OfficeList:
    LocList.append(DetectOffice(BoxR, Office,VertPoint))
    BoxR=FilterBox(BoxR,LocList,VertPoint)

#LeftPage
OfficeList=NxRows['Office'].tolist()
for Office in OfficeList:
    LocList.append(DetectOffice(BoxL, Office,VertPoint))
    BoxL=FilterBox(BoxL,LocList,VertPoint)

Dict[Page]=LocList

# Step C. Get Position Info

In [7]:
PosiDict,PosiDict_CLOVA={},{}

In [8]:
import itertools
#Split quardrant into offices and detect Positions
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Position')
from OrganizePosi import Split, SplitClova, Crop, CropClova
from DetectPosi import DetectPosi,DetectPosiClova,AggregatePosi

CrossWalk=pd.read_csv(origin+"Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
Positions=CrossWalk.tolist()
PosiDict['Pre']=[]
PosiDict_CLOVA['Pre']=[]

DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)
DF_CLOVA=CropClova(textsCLOVA,HoriPoint,VertPoint,Dict,Page)

##UR
BoxList,BoxListCLOVA=Split(DF['UR']['Box'],DF['UR']['Thres']),SplitClova(DF_CLOVA['UR']['Box'],DF_CLOVA['UR']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UR']['Thres'],PosiDict_CLOVA,Positions)

##LR
BoxList,BoxListCLOVA=Split(DF['LR']['Box'],DF['LR']['Thres']),SplitClova(DF_CLOVA['LR']['Box'],DF_CLOVA['LR']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LR']['Thres'],PosiDict_CLOVA,Positions)

##UL
BoxList,BoxListCLOVA=Split(DF['UL']['Box'][1:],DF['UL']['Thres']),SplitClova(DF_CLOVA['UL']['Box'],DF_CLOVA['UL']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UL']['Thres'],PosiDict_CLOVA,Positions)

#LL
BoxList,BoxListCLOVA=Split(DF['LL']['Box'],DF['LL']['Thres']),SplitClova(DF_CLOVA['LL']['Box'],DF_CLOVA['LL']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LL']['Thres'],PosiDict_CLOVA,Positions)

FinalPosiDict=AggregatePosi(PosiDict,PosiDict_CLOVA)


# Step D. Refine Position List & Update Vertical Line

In [9]:
from DetectPosi import RefPosiDict
FinalPosiDict=RefPosiDict(texts,FinalPosiDict)

from Layout import RefineVert
VertPoint2=RefineVert(img,LocList,FinalPosiDict,VertPoint,HoriPoint)

from DetectPosi import RefPosiDict2
FinalPosiDict=RefPosiDict2(FinalPosiDict,VertPoint,VertPoint2)
FinalPosiDict={Office: list(itertools.chain(*FinalPosiDict[Office])) for Office in PosiDict}

# Main Code

- Specify Quadrant.

- Extract position level texts.

    sub-dict from position dictionary.

- Assign texts to positions $\times$ offices.

- Organize texts to columns.

- Test Improvement by Japanese OCR.

In [10]:
DF=Crop(texts,HoriPoint,VertPoint2,Dict,Page)

In [11]:
from OrganizeName import FilterPosi, RenewPosiList,GetOffice
from Extract import ExtractName

- Detecting Columns

  1. Estimate rough table structure.
        Get column length information.

  2. Find first column.
 
  3. Estimate location of gap in a column.
 
  4. Define starting point of information as the end of each gap. 
     Seperate column into cells.

  5. Extract data.

In [12]:
Data={}
project='Off'
Data[str(Page)]=[]
#UpperRight
Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'UR','NA',img,project,origin,Year,Page))
#LowerRight
Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'LR','UR',img,project,origin,Year,Page))
#BottomLeft
Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'UL','LR',img,project,origin,Year,Page))
#BottomLeft
Data[str(Page)].append(ExtractName(DF,FinalPosiDict,HoriPoint,VertPoint2,'LL','UL',img,project,origin,Year,Page))

PrePosi
書記
雇
囑託員
Failed to find tables
主事
主事
書記
雇
Failed to find tables
主事
主事
書記
雇
Failed to detect top rows
技師
Failed to find tables
技師
Failed to find tables
技師
書記


In [21]:
Data['20'][0]['動員課']['主事'][0]['Names']

['日',
 '主事',
 '課長鹿谷義',
 '生活岡新掛長丸井玄信',
 '時局掛長入江博房',
 '杉岳成宗三ノ五五九',
 '陶體掛長鈴木由藏',
 '野町四番九ノ二']